In [ ]:
pip install ydata-profiling[pyspark]

In [55]:
from pyspark.sql import SparkSession
from ydata_profiling import ProfileReport

In [56]:
warehouse_location = 'hdfs://hdfs-nn:9000/warehouse'

spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Python Spark profiling example") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

In [57]:
# Copiar ficheiro para o layer bronze
from os import PathLike
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

from_path = "./amazon_titles.csv"
to_path = "/demo/bronze/amazon_titles.csv"

client.delete(to_path)
client.upload(to_path, from_path)

'/demo/bronze/amazon_titles.csv'

In [2]:
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/amazon_titles.csv"

In [6]:
# Ler ficheiro
amazon_credits_df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

In [ ]:
amazon_titles_df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

In [6]:
# Expor o schema e o conteúdo
amazon_titles_df.printSchema()
amazon_titles_df.show()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- age_certification: string (nullable = true)
 |-- runtime: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- seasons: double (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- imdb_score: double (nullable = true)
 |-- imdb_votes: double (nullable = true)
 |-- tmdb_popularity: double (nullable = true)
 |-- tmdb_score: double (nullable = true)

+--------+--------------------+-----+--------------------+------------+-----------------+-------+--------------------+--------------------+-------+---------+----------+----------+---------------+----------+
|      id|               title| type|         description|release_year|age_certification|runtime|              genres|production_countries|seasons|  imdb_id|imdb_

In [22]:
amazon_titles_df.select("seasons").distinct().count()

33

In [29]:
amazon_titles_df.filter(amazon_titles_df["imdb_score"].isNull()).count()

1021

In [42]:
amazon_titles_df.filter(amazon_titles_df["imdb_id"].isNull()).count()

667

In [43]:
amazon_titles_df.filter(amazon_titles_df["tmdb_popularity"].isNull()).count()

547

In [64]:
amazon_titles_df.filter(amazon_titles_df["imdb_votes"].isNull()).count()

1031

In [46]:
amazon_titles_df.select("tmdb_score").distinct().count()

90

In [18]:
from pyspark.sql import functions as F

# número total de linhas
total_rows = amazon_titles_df.count()

# para cada coluna, contar valores repetidos
for col in amazon_titles_df.columns:
    distinct_count = amazon_titles_df.select(col).distinct().count()
    repeated_count = total_rows - distinct_count
    print(f"{col}: {repeated_count} valores repetidos")

id: 3 valores repetidos
title: 134 valores repetidos
type: 9869 valores repetidos
description: 136 valores repetidos
release_year: 9761 valores repetidos
age_certification: 9859 valores repetidos
runtime: 9664 valores repetidos
genres: 7843 valores repetidos
production_countries: 9374 valores repetidos
seasons: 9838 valores repetidos
imdb_id: 669 valores repetidos
imdb_score: 9784 valores repetidos
imdb_votes: 6220 valores repetidos
tmdb_popularity: 4545 valores repetidos
tmdb_score: 9781 valores repetidos


In [21]:
from pyspark.sql import functions as F

# agrupa pelos títulos e conta quantas vezes aparecem
duplicados = (
    amazon_titles_df.groupBy("title")
    .agg(F.count("*").alias("ocorrencias"))
    .filter(F.col("ocorrencias") > 1)
)

# mostra um exemplo de filme repetido
duplicados.show(1)

+----------+-----------+
|     title|ocorrencias|
+----------+-----------+
|Black Gold|          2|
+----------+-----------+
only showing top 1 row



In [60]:
from pyspark.sql import functions as F
min_id = amazon_titles_df.agg(F.min("imdb_votes").alias("min_id")).collect()[0]["min_id"]
amazon_titles_df.select(F.min("imdb_votes"), F.max("imdb_votes")).show()

+---------------+---------------+
|min(imdb_votes)|max(imdb_votes)|
+---------------+---------------+
|            5.0|      1133692.0|
+---------------+---------------+



In [13]:
#In case of error select a subset of columns until you find the column that causes that.
#For start we can use describe as starting point for data profiling
#For this example the column summary was removed due to a conflit with the first describe column "summary"
amazon_titles_df.describe(['id','title','type','description','release_year','age_certification','runtime','genres','production_countries','seasons','imdb_id','imdb_score','imdb_votes','tmdb_popularity','tmdb_score']).toPandas()

,summary,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,count,9871,9871,9871,9752,9871,3384,9871,9871,9871,1357,9204,8850,8840,9324,7789
1,mean,None,6117.181818181818,None,None,2001.327221152872,None,85.97305237564584,None,None,2.791451731761238,None,5.976395480225987,8533.614253393665,6.91020441834339,5.984247015021188
2,stddev,None,16135.336840724347,None,None,25.810071382499803,None,33.51246623729997,None,None,4.148957721772592,None,1.3438415684624896,45920.151904649625,30.00409812190443,1.517985696177188
3,min,tm100001,#Home,MOVIE,"""A Night with Joshua Bassett"" was a one night ...",1912,G,1,"['action', 'animation', 'comedy', 'crime']","['AF', 'US']",1.0,tt0002199,1.1,5.0,1.1E-5,0.8
4,max,ts9913,부릉! 부릉! 브루미즈,SHOW,"Old Monk is Kannada means ""Hale Sanyasi"". Na...",2022,TV-Y7,549,[],[],51.0,tt9914192,9.9,1133692.0,1437.906,10.0


In [61]:
#Select the columns to profile. 
df_to_profile = amazon_titles_df.select("id","title","type","description","release_year","age_certification","runtime","genres","production_countries","seasons","imdb_id","imdb_score","imdb_votes","tmdb_popularity","tmdb_score")

In [62]:
#create the Profile report using the ydata profiling tool. More info at https://docs.profiling.ydata.ai/
report = ProfileReport(df_to_profile)

In [ ]:
report.to_file('amazon_titles.html')

In [12]:
#close spark session
spark.stop()